**Match Individual-Gene Loss Cohorts to REdiscoverTE**
This notebook loops over the individual-gene cohorts and checks which altered samples are present in the REdiscoverTE log2CPM ExpressionSet.

The key output for each gene/cohort is:

pancancer_<gene_set_label>_samples_with_RNA_match_TE_ready.csv
Those TE-ready files are the inputs for WT matching.

In [ ]:
# Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(Biobase)
library(stringr)
library(purrr)

project_root <- "/home/kennes38/ResearchProject"
notebook_dir <- file.path(project_root, "04_individualgenes_TEdata_overlap")
dir.create(notebook_dir, showWarnings = FALSE, recursive = TRUE)
setwd(notebook_dir)

loss_input_dir <- "../04_individualgenes_LoFHomDel_from_existingoutputs"

# Extracted REdiscoverTE data folder
tedata_root <- "../TEdata_resources"

matching_output_dir <- "."

dir.exists(loss_input_dir)
dir.exists(tedata_root)
dir.exists(matching_output_dir)

In [ ]:
# Define cohorts for processing

analysis_definitions <- tibble::tribble(
  ~gene_set_label, ~gene_set_group, ~display_name, ~genes,
  "kdm6a_loss",   "histone_modifier", "KDM6A loss",   list(c("KDM6A")),
  "kdm6b_loss",   "histone_modifier", "KDM6B loss",   list(c("KDM6B")),
  "nsd1_loss",    "histone_modifier", "NSD1 loss",    list(c("NSD1")),
  "nsd2_loss",    "histone_modifier", "NSD2 loss",    list(c("NSD2")),
  "setd2_loss",   "histone_modifier", "SETD2 loss",   list(c("SETD2")),
  "kdm6a_b_loss", "histone_modifier", "KDM6A/B loss", list(c("KDM6A", "KDM6B")),
  "nsd1_2_loss",  "histone_modifier", "NSD1/2 loss",  list(c("NSD1", "NSD2")),
  "arid1a_loss",   "swi_snf", "ARID1A loss",   list(c("ARID1A")),
  "arid1b_loss",   "swi_snf", "ARID1B loss",   list(c("ARID1B")),
  "arid1a_b_loss", "swi_snf", "ARID1A/B loss", list(c("ARID1A", "ARID1B")),
  "smarca4_loss",  "swi_snf", "SMARCA4 loss",  list(c("SMARCA4")),
  "smarcb1_loss", "swi_snf", "SMARCB1 loss", list(c("SMARCB1")),
  "pbrm1_loss",   "swi_snf", "PBRM1 loss",   list(c("PBRM1"))
)

analysis_definitions %>%
  mutate(genes_text = purrr::map_chr(genes, paste, collapse = ";")) %>%
  select(gene_set_label, gene_set_group, display_name, genes_text)

In [ ]:
# Load REdiscoverTE sample IDs

rds_hits <- list.files(
  tedata_root,
  pattern = "eset_TCGA_TE_intergenic_logCPM\\.RDS$",
  recursive = TRUE,
  full.names = TRUE
)

rds_hits
stopifnot(length(rds_hits) == 1)

dat <- readRDS(rds_hits[1])

# TEdata_full_id is the exact sample ID used in the ExpressionSet
tedata_samples <- tibble(
  TEdata_full_id = sampleNames(dat),
  sample_id_16 = substr(sampleNames(dat), 1, 16),
  patient_id = substr(sampleNames(dat), 1, 12),
  sample_type_code = substr(sampleNames(dat), 14, 15)
) %>%
  distinct()

cat("REdiscoverTE samples:", nrow(tedata_samples), "\n")
tedata_samples %>% count(sample_type_code, sort = TRUE)

In [ ]:
# Set helper function to match one cohort

match_one_cohort_to_tedata <- function(def_row) {
  gene_set_label <- def_row$gene_set_label
  message("===== ", gene_set_label, " =====")

  case_file <- file.path(loss_input_dir, paste0("pancancer_", gene_set_label, "_patients.csv"))

  if (!file.exists(case_file)) {
    warning("Missing input file for ", gene_set_label, ": ", case_file)
    return(tibble(gene_set = gene_set_label, n_loss_samples = 0, n_TE_ready = 0))
  }

  loss_cases <- read_csv(case_file, show_col_types = FALSE) %>%
    mutate(
      sample_id_16 = substr(as.character(sample_id_16), 1, 16),
      patient_id = substr(as.character(patient_id), 1, 12),
      case_sample_type_code = substr(sample_id_16, 14, 15)
    ) %>%
    distinct()

  loss_with_tedata_match <- loss_cases %>%
    left_join(
      tedata_samples %>% select(TEdata_full_id, TEdata_sample_id_16 = sample_id_16),
      by = c("sample_id_16" = "TEdata_sample_id_16")
    ) %>%
    mutate(has_TEdata_match = !is.na(TEdata_full_id))

  te_ready_cases <- loss_with_tedata_match %>%
    filter(has_TEdata_match) %>%
    arrange(project, sample_id_16)

  write_csv(
    loss_with_tedata_match,
    file.path(matching_output_dir, paste0("pancancer_", gene_set_label, "_samples_with_TEdata_match.csv"))
  )

  write_csv(
    te_ready_cases,
    file.path(matching_output_dir, paste0("pancancer_", gene_set_label, "_samples_with_RNA_match_TE_ready.csv"))
  )

  cat("Loss samples:", nrow(loss_cases), "\n")
  cat("TE-ready samples:", nrow(te_ready_cases), "\n")

  tibble(
    gene_set = gene_set_label,
    gene_set_display = def_row$display_name,
    genes_tested = paste(unique(unlist(def_row$genes, use.names = FALSE)), collapse = ";"),
    n_loss_samples = nrow(loss_cases),
    n_TE_ready = nrow(te_ready_cases),
    n_TE_ready_primary_tumour = sum(te_ready_cases$case_sample_type_code == "01", na.rm = TRUE)
  )
}

In [ ]:
# Run TEdata matching for all cohorts

definition_rows <- split(analysis_definitions, seq_len(nrow(analysis_definitions)))

te_match_summary <- purrr::map_dfr(definition_rows, match_one_cohort_to_tedata)

write_csv(
  te_match_summary,
  file.path(matching_output_dir, "individual_gene_TEdata_match_summary.csv")
)

te_match_summary %>% arrange(desc(n_TE_ready_primary_tumour))